In [8]:
###################################################################################
# Generate enviPath TPs prediction
###################################################################################
## version:1.1 - modified from https://github.com/enviPath/enviPath-python
## Date: 2026-09-03
## Authors: Boris Droz, Nick Cully 
## Modified by Nick Cully to improve documentation and code structure
################################################################################
## Description:
##  Create a suspect list of transformation products for given target compounds
##  YOU need to creat a input and output folder.
################################################################################

In [9]:
%pip install --upgrade enviPath-python pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
### load library and env.
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from os import getenv, path
from time import sleep
from dotenv import load_dotenv
from enviPath_python.enviPath import enviPath
from enviPath_python.objects import Pathway
from pathlib import Path
from dotenv import load_dotenv
from os import getenv
from requests.exceptions import Timeout, RequestException

import glob
import pandas as pd
import requests
import traceback

##### SETUP PARAMETER
workdir = Path(r"C:\Users\drozditb\Documents\GitHub\MS_Rtool\Suspect_list\input")
input_file = workdir / "input_suspect_testdata.csv"

#### LOGIN TO ENVIPATH
username ="Boris-Droz" #getenv("ENVIPATH_USER")
password = "FIELD_LABORATORY_PASSWORD" #getenv("ENVIPATH_PASS")

In [11]:
# ==============================================================================
# SECTION 1: LOGIN & DATA PREPARATION
# What it does: Connects to the enviPath online database using your account 
# credentials inside the env file, opens your default workspace package, 
# loads your input spreadsheet of chemical compounds, and removes any duplicate 
# chemical entries to save time.
# ==============================================================================

# --- 1. SETUP & AUTHENTICATION ---
env_path = workdir / ".env"
load_dotenv(dotenv_path=env_path, override=True)

input_df = pd.read_csv(input_file)

# Connect using the secure variables
eP = enviPath("https://envipath.org/api/legacy/", new_api=True)
eP.login(username, password)

print("Successfully logged in to enviPath.")

me = eP.who_am_i()
package = me.get_default_package()

print(f"{me}")

# Drops exact duplicate rows based on name or SMILES to save time
input_df = input_df.drop_duplicates(subset=["SMILES"]).reset_index(drop=True)

Successfully logged in to enviPath.
User: Boris-Droz (https://envipath.org/user/507498f7-3a75-437a-b41c-d565a21743c5)


In [12]:
# --- CACHING OPTIMIZATION: Skip already processed compounds for the sake of brevity---
processed_smiles = set()
existing_files = glob.glob(str(input_file))
if existing_files:
    latest_result_file = max(existing_files, key=path.getmtime)
    try:
        old_df = pd.read_csv(latest_result_file)
        if "parent_smiles" in old_df.columns:
            processed_smiles = set(old_df["parent_smiles"].dropna().unique())
            print(f"Found existing cache. Skipping {len(processed_smiles)} already processed parent compounds.")
    except Exception as e:
        print(f"Could not load cache: {e}")

# Filter out already processed rows if desired, or handle them inside the function
remaining_df = input_df[~input_df["SMILES"].isin(processed_smiles)].reset_index(drop=True)
print(f"Compounds remaining to process: {len(remaining_df)} out of {len(input_df)}")

Compounds remaining to process: 3 out of 3


In [13]:
# ==============================================================================
# SECTION 2: SINGLE CHEMICAL PROCESSING FUNCTION
# What it does: Defines the instructions for handling *one* chemical at a time. 
# It sends the chemical to enviPath to predict its breakdown pathway, waits for the 
# server to finish calculating, collects all the resulting child chemical nodes 
# (structure breakdown products), and formats them into a neat list.
# ==============================================================================

def safe_is_running(pw):
    try:
        return pw.is_running()
    except KeyError:
        return True  # treat malformed/incomplete status as "still running", then keep polling

all_results = []

print("Starting batch prediction...\n")
for index, row in remaining_df.iterrows():
    compound_name = row["name"]
    smiles = row["SMILES"]
    print(f"Processing [{index+1}/{len(remaining_df)}]: {compound_name} ({smiles})")

    try:
        pw = Pathway.create(package, smiles=smiles, name=compound_name)

        max_attempts = 60
        attempt = 0

        print(f"  -> Waiting for prediction to finish", end="", flush=True)

        while attempt < max_attempts:
            pw.refresh()
            if not safe_is_running(pw):
                break
            # prints out a sequential dot to indicate waiting
            print(".", end="", flush=True)
            sleep(3)
            attempt += 1

        if pw.is_running():
            print(f"  -> Timed out waiting for {compound_name}, skipping")
            continue
        if pw.has_failed():
            print(f"  -> Prediction failed for {compound_name}")
            continue

        nodes = pw.get_nodes()       
        pathway_url = pw.get_id()       
        edges = pw.get_edges()
        reaction = edges[0].get_reaction()
        print(f"\n")
        if reaction:
            print(reaction.get_id())
            print(reaction.get_name())
        
        # Map each end-node ID to the reaction that produced it
        node_to_reaction = {}
        for edge in edges:
            
            reaction_name = edge.get_reaction_name()
            reaction_id = edge.get_id()  # or edge.get_reaction() if that returns a Reaction object with its own get_id()            
            for end_node in edge.get_end_nodes():
                node_to_reaction[end_node.get_id()] = (reaction_name, reaction_id)

        
        for node in nodes:
            node_smiles = node.get_smiles()
            depth = node.get_depth()
            node_id = node.get_id()
            
            reaction_name, reaction_id = node_to_reaction.get(node_id, (None, None))

            all_results.append({
                'Pathway URL': pathway_url,
                'SMILES': node_smiles,
                'depth': depth,
                'reaction_name': reaction_name,
                'reaction_id': reaction_id,
                'parent_smiles': smiles
            })
            
    except KeyboardInterrupt:
        print("\nProcess interrupted by user.")
        break
    except Exception as e:
        print(f"  -> Error processing {compound_name}: {e}")
        traceback.print_exc()

Starting batch prediction...

Processing [1/3]: Atrazine (CCNC1=NC(=NC(=N1)Cl)NC(C)C)
  -> Waiting for prediction to finish....

https://envipath.org/package/4ec6668f-e864-4fad-9bb3-50b9d66a5b8b/reaction/01036a3b-be13-4eaa-a1ef-00725de53d48
Reaction 55
Processing [2/3]: S-Metolachlor (CCC1=CC=CC(=C1N([C@@H](C)COC)C(=O)CCl)C)
  -> Waiting for prediction to finish...

https://envipath.org/package/4ec6668f-e864-4fad-9bb3-50b9d66a5b8b/reaction/21566967-0845-4797-bfca-0465f084c635
Reaction 89
Processing [3/3]: 6:2 fluorotelomer sulfonamido propyl methyl amine (CNCCCNS(=O)(=O)CCC(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)(F)F)
  -> Waiting for prediction to finish...

https://envipath.org/package/4ec6668f-e864-4fad-9bb3-50b9d66a5b8b/reaction/cb093bef-67b4-49ab-865e-0bfcfc8f85c6
Reaction 109


In [15]:
# ==============================================================================
# SECTION 3: MULTITASKING / BATCH RUNNER
# Consolidates this run's results with any cached prior results 
# and writes the combined CSV
# ==============================================================================

# Combine new results with cache if any existed, otherwise save fresh
new_results_df = pd.DataFrame(all_results) if all_results else pd.DataFrame()

if not new_results_df.empty:
    if existing_files and len(processed_smiles) > 0:
        final_df = pd.concat([old_df, new_results_df], ignore_index=True)
    else:
        final_df = new_results_df
        
    timestamp = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_filename = f"output/prediction_envipath_{timestamp}.csv"
    final_df.to_csv(output_filename, index=False)
    print(f"\nBatch complete! Saved results to {output_filename} with {len(final_df)} total rows.")
else:
    print("\nNo new results generated (or all compounds were already cached).")


Batch complete! Saved results to output/prediction_envipath_2026-09-22_14-26-37.csv with 57 total rows.
